# Prompt Kaynaklari Wiki Sync

Agent-description/agent-mimarisi (single-agent vs. multi-agent, sequential vs.
parallel) akisi icin secilen 6 statik referans kaynagini (bkz. AGENTS.md)
ceker, ozetler ve Azure DevOps Wiki'ye (`/Prompt-Kaynaklari/...`) yazar. Bu
sayfalar daha sonra ilgili agent'in system prompt'una statik olarak gomulmek
uzere kullanilir.

Kaynaklar:
1. Anthropic - Building Effective Agents (vendor) — genel workflow/agent
   taksonomisi (prompt-chaining, routing, parallelization,
   orchestrator-workers, evaluator-optimizer).
2. Anthropic - How We Built Our Multi-Agent Research System (vendor) —
   ne zaman multi-agent'a gecilmeli, orchestrator-worker delegasyonu.
3. Google ADK - Sequential Agents (vendor) — sadece **sequential**
   (sirali) agent zincirleme deseni, `SequentialAgent` ornegi.
4. OpenAI Agents SDK - Agent Orchestration (vendor) — kod-tabanli
   sequential chaining ile paralel calistirmanin (asyncio.gather) acik
   karsilastirmasi.
5. Microsoft Semantic Kernel - Sequential Orchestration (vendor) —
   `SequentialOrchestration` sinifi, agent pipeline (analyst -> copywriter
   -> editor) ornegi.
6. Microsoft Azure Architecture Center - AI Agent Orchestration Patterns
   (vendor) — "Start with the right level of complexity" karar tablosu
   (direct model call / single agent with tools / multiagent), 5 pattern
   (sequential, concurrent, group chat, handoff, magentic) icin "when to
   use / when to avoid" listeleri, pattern karsilastirma tablosu, ortak
   antipattern'lar, maliyet/guvenlik/gozlemlenebilirlik notlari.

3-6 numarali kaynaklar, agent'in mimari onerisi verirken sadece paralel
(orchestrator-workers) degil, **sirali (sequential) agent zincirleme**
secenegini de kaynakli/dogrulanmis sekilde degerlendirebilmesi icin
eklendi. 6 numarali kaynak ozellikle "ne zaman single-agent yeterli, ne
zaman multi-agent'a (ve hangi patterne) gecilmeli" sorusuna dogrudan
cevap veren tek kaynak — Optimizer'in karar agacindaki en kritik bosluk
buydu.

robots.txt ve lisans kontrolu onceden yapildi (bkz. AGENTS.md):
- anthropic.com: `robots.txt` -> `User-Agent: *` / `Allow: /` (serbest)
- learn.microsoft.com: `robots.txt` -> ilgili path'lar
  (`/en-us/semantic-kernel/...`, `/en-us/azure/architecture/...`)
  Disallow listesinde yok (serbest)
- google.github.io, openai.github.io: `robots.txt` yok (404 -> serbest
  kabul edilir, `check_robots_allowed` zaten 4xx'te True donuyor)


In [0]:
%pip install beautifulsoup4

In [0]:
%run "./Utils"

## Kaynak tanimlari

In [0]:
import json
from datetime import datetime, timezone

from bs4 import BeautifulSoup


ARTICLE_SOURCES = [
    {
        "name": "Anthropic - Building Effective Agents",
        "category": "vendor",
        "url": "https://www.anthropic.com/engineering/building-effective-agents",
        "license": "Anthropic PBC, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Anthropic-Building-Effective-Agents",
    },
    {
        "name": "Anthropic - How We Built Our Multi-Agent Research System",
        "category": "vendor",
        "url": "https://www.anthropic.com/engineering/multi-agent-research-system",
        "license": "Anthropic PBC, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Anthropic-Multi-Agent-Research-System",
    },
    {
        "name": "Google ADK - Sequential Agents",
        "category": "vendor",
        "url": "https://google.github.io/adk-docs/agents/workflow-agents/sequential-agents/",
        "license": "Apache License 2.0 (Google, adk-docs). Dahili referans amacli.",
        "wiki_path": "/Prompt-Kaynaklari/Google-ADK-Sequential-Agents",
    },
    {
        "name": "OpenAI Agents SDK - Agent Orchestration",
        "category": "vendor",
        "url": "https://openai.github.io/openai-agents-python/multi_agent/",
        "license": "OpenAI, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/OpenAI-Agents-SDK-Orchestration",
    },
]

MS_LEARN_SOURCES = [
    {
        "name": "Microsoft Semantic Kernel - Sequential Orchestration",
        "category": "vendor",
        "url": "https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-orchestration/sequential",
        "license": "Microsoft, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Microsoft-Semantic-Kernel-Sequential-Orchestration",
    },
    {
        "name": "Microsoft Azure Architecture Center - AI Agent Orchestration Patterns",
        "category": "vendor",
        "url": "https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns",
        "license": "Microsoft, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Microsoft-Azure-Architecture-Center-Agent-Patterns",
    },
]

ALL_SOURCES = ARTICLE_SOURCES + MS_LEARN_SOURCES


## robots.txt dogrulamasi (calisma zamaninda)

In [0]:
for source in ALL_SOURCES:
    allowed = check_robots_allowed(source["url"])
    print(f"{source['name']}: {'ALLOWED' if allowed else 'DISALLOWED'}")


## Genel amacli makale/dokumantasyon icerik cikarma

`anthropic.com/engineering/...`, `google.github.io/adk-docs/...` ve
`openai.github.io/openai-agents-python/...` sayfalarinin ucu de sunucu
tarafinda tam HTML olarak render ediliyor ve govdeleri bir `<article>`
etiketi icinde okunabilir `<h2>/<h3>/<p>/<li>` etiketleriyle geliyor (ADK ve
OpenAI Agents SDK dokumantasyonu ayni mkdocs-material temasini kullaniyor).
Bu yuzden hepsi icin tek bir BeautifulSoup tabanli fonksiyon yeterli. Kod
ornekleri (`<pre>` bloklari, orn. `SequentialAgent(sub_agents=[...])`)
fenced ```` ``` ```` bloklarina, karsilastirma tablolari (`<table>`, orn.
OpenAI Agents SDK'daki 'agents as tools vs handoffs' tablosu) `|`
ile ayrilmis satirlara cevrilerek korunuyor — sadece prose cikarilsa bu
yapisal bilgi kaybolurdu.


In [0]:
def extract_list_item_text(list_item):

    parts = []

    for child in list_item.children:
        if getattr(child, "name", None) in ("ul", "ol", "pre"):
            continue
        if isinstance(child, str):
            parts.append(child)
        else:
            parts.append(child.get_text(" ", strip=True))

    return " ".join(part.strip() for part in parts if part.strip())


def extract_table_text(table):

    rows = []

    for row in table.find_all("tr"):
        cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
        if cells:
            rows.append(" | ".join(cells))

    return "\n".join(rows)


def render_content_elements(elements):

    lines = []

    for element in elements:

        if element.name == "pre":
            code_text = element.get_text().strip()
            if code_text:
                lines.append(f"```\n{code_text}\n```")
            continue

        if element.name == "table":
            table_text = extract_table_text(element)
            if table_text:
                lines.append(table_text)
            continue

        if element.name == "li":
            text = extract_list_item_text(element)
        else:
            text = element.get_text(" ", strip=True)

        if not text:
            continue

        if element.name == "h2":
            lines.append(f"\n## {text}\n")
        elif element.name == "h3":
            lines.append(f"\n### {text}\n")
        elif element.name == "h4":
            lines.append(f"\n#### {text}\n")
        elif element.name == "li":
            lines.append(f"- {text}")
        else:
            lines.append(text)

    return "\n\n".join(line.strip() for line in lines if line.strip())


def extract_engineering_article_content(raw_html):

    soup = BeautifulSoup(raw_html, "html.parser")
    article = soup.find("article")

    if article is None:
        return ""

    elements = article.find_all(["h2", "h3", "h4", "p", "li", "pre", "table"])

    return render_content_elements(elements)


## Microsoft Learn icerik cikarma

Microsoft Learn dokumantasyonu `<article>` etiketi kullanmiyor; govde
birden fazla `div.content` blogu icinde geliyor (ilki sadece `<h1>`, asil
icerik en buyugunde). Bazi sayfalar (orn. Semantic Kernel Sequential
Orchestration) C#/Python/Java icin ayni ornegi 3 kez tekrarliyor
(`div.zone[data-pivot="programming-language-..."]`) — projede zaten
Python kullanildigi icin sadece `programming-language-python` pivot'u
tutulup diger ikisi (csharp, java) `decompose()` ile silinerek
tekrar/gurultu onleniyor. Azure Architecture Center sayfasi gibi pivot
icermeyen sayfalarda bu adim etkisiz kalir (decompose edilecek zone
bulunmaz), fonksiyon degismeden calisir.


In [0]:
def extract_ms_learn_content(raw_html, keep_pivot="programming-language-python"):

    soup = BeautifulSoup(raw_html, "html.parser")
    content_divs = soup.find_all("div", class_="content")

    if not content_divs:
        return ""

    main_content = max(content_divs, key=lambda tag: len(tag.get_text()))

    for zone in main_content.find_all("div", class_="zone"):
        pivot = zone.get("data-pivot")
        if pivot and pivot != keep_pivot:
            zone.decompose()

    elements = main_content.find_all(["h2", "h3", "h4", "p", "li", "pre", "table"])

    return render_content_elements(elements)


## Agent-friendly wiki icerik olusturucu

In [0]:
def build_reference_wiki_content(source_name, source_url, license_text, fetched_at, body_text):

    metadata = {
        "source_name": source_name,
        "source_url": source_url,
        "license": license_text,
        "fetched_at": fetched_at,
        "purpose": "agent-description/agent-mimarisi (single-agent, sequential, multi-agent) akisi icin statik referans kaynagi",
    }

    metadata_block = json.dumps(metadata, ensure_ascii=False, indent=2)

    return (
        f"# {source_name}\n\n"
        f"```json\n{metadata_block}\n```\n\n"
        f"## Icerik\n\n{body_text}\n"
    )


## Wiki'ye yazma

In [0]:
fetched_at = datetime.now(timezone.utc).isoformat()

for source in ARTICLE_SOURCES:

    raw_html = fetch_url_text(source["url"])
    body_text = extract_engineering_article_content(raw_html)

    content = build_reference_wiki_content(
        source["name"],
        source["url"],
        source["license"],
        fetched_at,
        body_text,
    )

    push_wiki_page(source["wiki_path"], content)


In [0]:
for source in MS_LEARN_SOURCES:

    ms_learn_raw_html = fetch_url_text(source["url"])
    ms_learn_body = extract_ms_learn_content(ms_learn_raw_html)

    ms_learn_content = build_reference_wiki_content(
        source["name"],
        source["url"],
        source["license"],
        fetched_at,
        ms_learn_body,
    )

    push_wiki_page(source["wiki_path"], ms_learn_content)


## Index sayfasi

In [0]:
index_content = (
    "# Prompt Yazma Referans Kaynaklari\n\n"
    "Agent-description/agent-mimarisi (single-agent, sequential, multi-agent) "
    "akisi icin statik olarak gomulen 6 kaynak:\n\n"
    "1. [Anthropic - Building Effective Agents](/Prompt-Kaynaklari/Anthropic-Building-Effective-Agents)\n"
    "2. [Anthropic - How We Built Our Multi-Agent Research System](/Prompt-Kaynaklari/Anthropic-Multi-Agent-Research-System)\n"
    "3. [Google ADK - Sequential Agents](/Prompt-Kaynaklari/Google-ADK-Sequential-Agents)\n"
    "4. [OpenAI Agents SDK - Agent Orchestration](/Prompt-Kaynaklari/OpenAI-Agents-SDK-Orchestration)\n"
    "5. [Microsoft Semantic Kernel - Sequential Orchestration](/Prompt-Kaynaklari/Microsoft-Semantic-Kernel-Sequential-Orchestration)\n"
    "6. [Microsoft Azure Architecture Center - AI Agent Orchestration Patterns](/Prompt-Kaynaklari/Microsoft-Azure-Architecture-Center-Agent-Patterns)\n\n"
    f"Son senkronizasyon: {fetched_at}\n"
)

push_wiki_page("/Prompt-Kaynaklari", index_content)
